# Embedding + Classifier Pipeline (General Purpose)

A reusable notebook for evaluating text classification pipelines based on transformer embeddings + traditional ML classifiers.

**Designed for:**
- Swapping in any dataset (HateXplain, HateBenchSet, SGHateCheck, synthetic SG data)
- Swapping in any encoder (pre-trained or fine-tuned, CLS or rationale pooling)
- Training LogReg / XGBoost / MLP and comparing results
- Cross-dataset evaluation: train on dataset A, test on dataset B

**Typical use cases:**
1. Evaluate how well HateXplain-trained models generalise to HateBenchSet (OOD English)
2. Evaluate the same models on SGHateCheck (OOD + language shift)
3. Add synthetic SG examples to training and measure if SGHateCheck accuracy improves

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Encoder to use for embedding extraction.
# Any HuggingFace model path or local directory works here.
# Examples:
#   'roberta-base'                              — pre-trained, no task signal
#   'bert-base-uncased'                         — pre-trained BERT
#   'bert-binary-hatexplain-finetuned'          — local fine-tuned binary
#   'shareeep/bert-base-fine-tuned-hatexplain'  — HF fine-tuned 3-class
#   'shareeep/roberta-base-fine-tuned-hatexplain'
ENCODER_MODEL   = 'roberta-base'
ENCODER_FAMILY  = 'roberta'          # 'bert' or 'roberta' — controls tokenizer class
POOLING         = 'cls'              # 'cls' | 'mean' | 'rationale'

# Training dataset — a list of dicts with keys 'text', 'label', and optionally
# 'rationale_scores' (list of floats, one per whitespace token).
# Populate TRAIN_DATA and TEST_DATA below, or use the loaders in the next cell.
TRAIN_DATA = []   # filled by data loading cells
TEST_DATA  = []   # filled by data loading cells

# Optional: path to pre-computed embeddings (numpy .npy pair or pkl dict).
# If set and the file exists, encoding is skipped.
# Set to None to always re-encode.
PRECOMPUTED_TRAIN_EMBEDDINGS = None   # e.g. 'embeddings/X_train_roberta.npy'
PRECOMPUTED_TEST_EMBEDDINGS  = None   # e.g. 'embeddings/X_test_roberta.npy'
PRECOMPUTED_TRAIN_LABELS     = None   # e.g. 'embeddings/y_train.npy'
PRECOMPUTED_TEST_LABELS      = None   # e.g. 'embeddings/y_test.npy'

# Label map — maps integer label → human-readable string for reports.
# Adjust to match your dataset.
LABEL_MAP = {0: 'hate', 1: 'offensive', 2: 'normal'}   # 3-class HateXplain
# LABEL_MAP = {0: 'hate', 1: 'non-hate'}                # binary

print("Config set.")
print(f"  Encoder : {ENCODER_MODEL}  [{ENCODER_FAMILY}, pooling={POOLING}]")
print(f"  Labels  : {LABEL_MAP}")

In [4]:
import json
import os
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModel,
    RobertaTokenizer, RobertaModel,
    BertTokenizer, BertModel,
)

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from xgboost import XGBClassifier
from copy import deepcopy

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## Data Loading

Run whichever loader matches your dataset. Each loader populates `TRAIN_DATA` and `TEST_DATA` as lists of dicts:
```python
{'text': str, 'label': int, 'rationale_scores': list[float] | None}
```
Only `text` and `label` are required. `rationale_scores` is used when `POOLING = 'rationale'`.

In [5]:
# ── LOADER A: HateXplain (JSONL) ──────────────────────────────────────────────
# Standard format used in this project. Each line is a JSON object.
# Label mapping: 0=hate, 1=offensive, 2=normal (set LABEL_MAP accordingly).

def load_hatexplain_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            item = json.loads(line)
            tokens = item['post_tokens']
            text = ' '.join(tokens)
            labels = item['annotators']['label']
            majority_label = max(set(labels), key=labels.count)

            # Rationale scores: average across annotators, aligned to token count
            rationales = item.get('rationales', [])
            valid = [r for r in rationales if len(r) == len(tokens)]
            if valid:
                rationale_scores = np.mean(valid, axis=0).tolist()
            else:
                rationale_scores = [0.0] * len(tokens)

            records.append({
                'text': text,
                'label': majority_label,
                'rationale_scores': rationale_scores,
            })
    return records

TRAIN_DATA = load_hatexplain_jsonl('hatexplain/hatexplain_train.jsonl')
VALID_DATA = load_hatexplain_jsonl('hatexplain/hatexplain_validation.jsonl')
TEST_DATA  = load_hatexplain_jsonl('hatexplain/hatexplain_test.jsonl')

# Combine train + valid for final training (common practice once hyperparams locked)
# TRAIN_DATA = TRAIN_DATA + VALID_DATA  # uncomment to use all labelled data

print(f"HateXplain loaded — train: {len(TRAIN_DATA)}, valid: {len(VALID_DATA)}, test: {len(TEST_DATA)}")
print(f"Label distribution (train): {pd.Series([r['label'] for r in TRAIN_DATA]).value_counts().to_dict()}")

HateXplain loaded — train: 15383, valid: 1922, test: 1924
Label distribution (train): {1: 6251, 0: 4748, 2: 4384}


In [ ]:
# ── LOADER B: HateBenchSet (CSV) ──────────────────────────────────────────────
# OOD generalization test on English data.
# Assumes a CSV with at least 'text' and 'label' columns.
# Update column names and label encoding to match your file.

# HATEBENCHSET_PATH = 'hatebenchset/hatebenchset.csv'   # update path as needed
# HATEBENCHSET_LABEL_COL  = 'label'
# HATEBENCHSET_TEXT_COL   = 'text'
# HATEBENCHSET_LABEL_MAP  = {'hate': 0, 'non-hate': 1}  # adjust to your CSV values

# def load_csv_dataset(path, text_col, label_col, label_map):
#     df = pd.read_csv(path)
#     records = []
#     for _, row in df.iterrows():
#         raw_label = row[label_col]
#         label = label_map.get(str(raw_label).strip().lower(), -1)
#         if label == -1:
#             continue  # skip rows with unmapped labels
#         records.append({'text': str(row[text_col]), 'label': label, 'rationale_scores': None})
#     return records

# HATEBENCKSET_DATA = load_csv_dataset(
#     HATEBENCHSET_PATH, HATEBENCHSET_TEXT_COL,
#     HATEBENCHSET_LABEL_COL, HATEBENCHSET_LABEL_MAP
# )
# print(f"HateBenchSet loaded: {len(HATEBENCKSET_DATA)} samples")

print("HateBenchSet loader ready (uncomment to use)")

In [ ]:
# ── LOADER C: SGHateCheck (CSV / JSONL) ───────────────────────────────────────
# Singapore-specific test benchmark.
# SGHateCheck has columns: 'test_case', 'label_gold', 'functionality', 'target_ident'
# Labels: 'hateful' / 'non-hateful'
# Binary task only — map to {0: hate, 1: non-hate} to match binary model.
# This dataset is EVALUATION ONLY — do not add to training.

# SGHATECHECK_PATH = 'sghatecheck/en_SGHateCheck.csv'   # update path after download
# SGHATECHECK_LABEL_MAP = {'hateful': 0, 'non-hateful': 1}

# def load_sghatecheck(path, label_map):
#     df = pd.read_csv(path)
#     records = []
#     for _, row in df.iterrows():
#         label = label_map.get(str(row['label_gold']).strip().lower(), -1)
#         if label == -1:
#             continue
#         records.append({
#             'text':             str(row['test_case']),
#             'label':            label,
#             'rationale_scores': None,
#             'functionality':    row.get('functionality', ''),
#             'target_ident':     row.get('target_ident', ''),
#         })
#     return records

# SGCHECK_DATA = load_sghatecheck(SGHATECHECK_PATH, SGHATECHECK_LABEL_MAP)
# print(f"SGHateCheck loaded: {len(SGCHECK_DATA)} samples")

print("SGHateCheck loader ready (uncomment to use)")

In [ ]:
# ── LOADER D: Synthetic / Custom Data ─────────────────────────────────────────
# For generated SG examples or any ad-hoc labelled text.
# Just build a list of dicts with 'text' and 'label'.

# Example: small hand-labelled SG dataset
# SYNTHETIC_SG = [
#     {'text': 'those bangla workers always cause trouble lah', 'label': 0, 'rationale_scores': None},
#     {'text': 'curry smell from neighbour unit very strong sia', 'label': 1, 'rationale_scores': None},
#     {'text': 'hdb void deck gathering again, damn noisy one',  'label': 2, 'rationale_scores': None},
# ]

# To augment training with synthetic data:
# TRAIN_DATA = TRAIN_DATA + SYNTHETIC_SG

print("Synthetic/custom loader ready (uncomment to use)")

## Encoder Setup + Embedding Extraction

Loads the encoder specified in Config. Supports CLS, mean, and rationale pooling for both BERT and RoBERTa families.

In [ ]:
# Load encoder — works for any HuggingFace model, local or remote
tokenizer = AutoTokenizer.from_pretrained(ENCODER_MODEL)
encoder   = AutoModel.from_pretrained(ENCODER_MODEL).to(DEVICE)
encoder.eval()
print(f"Loaded: {ENCODER_MODEL}")

In [ ]:
def get_embedding(text, rationale_scores=None):
    """Extract a 768-dim embedding using the globally configured encoder and pooling strategy."""
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding=True,
    )
    inputs_on_device = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = encoder(**inputs_on_device)

    hidden = outputs.last_hidden_state          # (1, seq_len, 768)
    attention_mask = inputs['attention_mask']   # (1, seq_len)

    if POOLING == 'cls':
        return hidden[0, 0, :].cpu().numpy()

    elif POOLING == 'mean':
        mask = attention_mask[0].unsqueeze(-1).float().to(DEVICE)
        summed = (hidden[0] * mask).sum(dim=0)
        return (summed / mask.sum()).cpu().numpy()

    elif POOLING == 'rationale':
        # Build subword-aligned weight vector from word-level rationale scores
        words = text.split()
        word_ids = []
        for word_idx, word in enumerate(words):
            n_sub = len(tokenizer.encode(word, add_special_tokens=False))
            word_ids.extend([word_idx] * n_sub)

        max_sub = inputs['input_ids'].shape[1] - 2
        word_ids_trimmed = word_ids[:max_sub]

        scores = rationale_scores if (rationale_scores and any(s > 0 for s in rationale_scores)) else None
        weights = [
            (scores[w] if scores and w < len(scores) else 1.0)
            for w in word_ids_trimmed
        ]
        seq_len = inputs['input_ids'].shape[1]
        full_weights = [0.0] + weights + [0.0] * (seq_len - len(weights) - 1)
        full_weights = full_weights[:seq_len]

        w_tensor = torch.tensor(full_weights, dtype=torch.float32).to(DEVICE)
        w_tensor = w_tensor * attention_mask[0].float().to(DEVICE)
        w_sum = w_tensor.sum()

        if w_sum == 0:
            mask = attention_mask[0].unsqueeze(-1).float().to(DEVICE)
            summed = (hidden[0] * mask).sum(dim=0)
            return (summed / mask.sum()).cpu().numpy()

        w = w_tensor.unsqueeze(-1)
        return ((hidden[0] * w).sum(dim=0) / w_sum).cpu().numpy()

    else:
        raise ValueError(f"Unknown pooling '{POOLING}'. Use 'cls', 'mean', or 'rationale'.")


def encode_records(records, desc='Encoding'):
    """Encode a list of {'text', 'label', 'rationale_scores'} dicts → X, y arrays."""
    X, y = [], []
    for rec in tqdm(records, desc=desc):
        emb = get_embedding(rec['text'], rec.get('rationale_scores'))
        X.append(emb)
        y.append(rec['label'])
    return np.array(X), np.array(y)

print("Embedding functions defined.")

In [ ]:
# ── Encode train + test (or load from disk if pre-computed) ───────────────────
def _try_load(x_path, y_path):
    if x_path and y_path and os.path.exists(x_path) and os.path.exists(y_path):
        print(f"  Loading pre-computed embeddings from {x_path}")
        return np.load(x_path), np.load(y_path)
    return None, None

X_train, y_train = _try_load(PRECOMPUTED_TRAIN_EMBEDDINGS, PRECOMPUTED_TRAIN_LABELS)
X_test,  y_test  = _try_load(PRECOMPUTED_TEST_EMBEDDINGS,  PRECOMPUTED_TEST_LABELS)

if X_train is None:
    print("Encoding training data...")
    X_train, y_train = encode_records(TRAIN_DATA, desc='Train')

if X_test is None:
    print("Encoding test data...")
    X_test, y_test = encode_records(TEST_DATA, desc='Test')

print(f"\nX_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")
print(f"Label distribution (train): {pd.Series(y_train).value_counts().to_dict()}")

## Classifier Training + Evaluation

In [ ]:
target_names = [LABEL_MAP[k] for k in sorted(LABEL_MAP)]

CLASSIFIERS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42,
                                              class_weight='balanced'),
    'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.05,
                                         max_depth=6, eval_metric='mlogloss',
                                         random_state=42),
    'MLP (256→128→64)':    MLPClassifier(hidden_layer_sizes=(256, 128, 64),
                                          activation='relu', max_iter=500,
                                          random_state=42, early_stopping=True,
                                          validation_fraction=0.1),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42,
                                                   class_weight='balanced'),
}

results = {}
for name, clf in CLASSIFIERS.items():
    print(f"\n{'─'*55}")
    print(f"  {name}")
    print(f"{'─'*55}")
    m = deepcopy(clf)
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    print(classification_report(y_test, y_pred, target_names=target_names))
    results[name] = {'model': m, 'acc': acc, 'macro_f1': macro_f1, 'preds': y_pred}

print("\n" + "="*55)
print("SUMMARY")
print("="*55)
summary_df = pd.DataFrame([
    {'Classifier': k, 'Accuracy': v['acc'], 'Macro F1': v['macro_f1']}
    for k, v in results.items()
]).sort_values('Macro F1', ascending=False)
print(summary_df.to_string(index=False))

## Cross-Dataset Evaluation

Train on one dataset, evaluate on another.  
Use this to measure OOD generalisation — e.g. train on HateXplain, test on HateBenchSet or SGHateCheck.

In [ ]:
def evaluate_on_dataset(records, clf, desc='OOD eval', label_map=None):
    """Encode a new dataset and evaluate a pre-trained classifier on it.

    Parameters
    ----------
    records   : list of {'text', 'label', 'rationale_scores'} dicts
    clf       : a fitted sklearn-compatible classifier
    desc      : label for the tqdm bar and output header
    label_map : dict {int: str} for the classification report

    Returns a dict with accuracy, macro_f1, and the predictions.
    """
    X_ood, y_ood = encode_records(records, desc=desc)
    y_pred = clf.predict(X_ood)
    acc      = accuracy_score(y_ood, y_pred)
    macro_f1 = f1_score(y_ood, y_pred, average='macro')

    tn = (label_map or LABEL_MAP)
    tnames = [tn[k] for k in sorted(tn)]
    print(f"\n── {desc} ──")
    print(f"Accuracy: {acc:.4f}  Macro F1: {macro_f1:.4f}")
    print(classification_report(y_ood, y_pred, target_names=tnames))
    return {'acc': acc, 'macro_f1': macro_f1, 'preds': y_pred, 'true': y_ood}


# ── Example: evaluate best in-distribution model on HateBenchSet ─────────────
# Uncomment after loading HATEBENCKSET_DATA in Loader B above.

# best_clf = results['Logistic Regression']['model']
# hb_results = evaluate_on_dataset(HATEBENCKSET_DATA, best_clf, desc='HateBenchSet OOD')

# ── Example: evaluate on SGHateCheck ─────────────────────────────────────────
# Uncomment after loading SGCHECK_DATA in Loader C above.
# Note: SGHateCheck is binary — make sure LABEL_MAP matches {0: 'hate', 1: 'non-hate'}

# sg_results = evaluate_on_dataset(SGCHECK_DATA, best_clf, desc='SGHateCheck OOD',
#                                  label_map={0: 'hate', 1: 'non-hate'})

print("Cross-dataset evaluation helpers ready (uncomment to run)")

## Per-Functionality Breakdown (SGHateCheck)

SGHateCheck annotates each test case with a `functionality` tag (e.g. `dehumanisation`, `threatening_language`) and `target_ident` (e.g. `race_chinese`, `religion_muslim`). This breakdown shows which categories the model struggles on — the most useful signal for targeted improvement.

In [ ]:
# Uncomment after running the SGHateCheck OOD eval above.
# Requires SGCHECK_DATA and sg_results from the previous cell.

# sg_df = pd.DataFrame(SGCHECK_DATA)
# sg_df['pred'] = sg_results['preds']
# sg_df['correct'] = (sg_df['pred'] == sg_df['label']).astype(int)

# # Accuracy per functionality
# func_acc = sg_df.groupby('functionality')['correct'].mean().sort_values()
# print("=== Accuracy by functionality ===")
# print(func_acc.to_string())

# # Accuracy per target identity group
# target_acc = sg_df.groupby('target_ident')['correct'].mean().sort_values()
# print("\n=== Accuracy by target identity ===")
# print(target_acc.to_string())

print("Per-functionality breakdown ready (uncomment after SGHateCheck OOD eval)")

## Save Embeddings to Disk

Run this once after encoding to avoid re-running RoBERTa inference next time. Saved arrays can be pointed to via `PRECOMPUTED_*` in Config.

In [ ]:
os.makedirs('embeddings', exist_ok=True)

# Build a safe filename tag from the encoder name and pooling strategy
_tag = ENCODER_MODEL.split('/')[-1].replace('-', '_') + f'_{POOLING}'

train_x_path = f'embeddings/X_train_{_tag}.npy'
train_y_path = f'embeddings/y_train_{_tag}.npy'
test_x_path  = f'embeddings/X_test_{_tag}.npy'
test_y_path  = f'embeddings/y_test_{_tag}.npy'

np.save(train_x_path, X_train)
np.save(train_y_path, y_train)
np.save(test_x_path,  X_test)
np.save(test_y_path,  y_test)

print(f"Saved:")
print(f"  {train_x_path}  {X_train.shape}")
print(f"  {test_x_path}   {X_test.shape}")
print(f"\nTo skip re-encoding next run, set in Config:")
print(f"  PRECOMPUTED_TRAIN_EMBEDDINGS = '{train_x_path}'")
print(f"  PRECOMPUTED_TRAIN_LABELS     = '{train_y_path}'")
print(f"  PRECOMPUTED_TEST_EMBEDDINGS  = '{test_x_path}'")
print(f"  PRECOMPUTED_TEST_LABELS      = '{test_y_path}'")

---

## Route B: Shallow Classifiers on Fine-Tuned Embeddings

Loads pre-computed embeddings from `embeddings/embedding_comparison_arrays.pkl` (saved by `embedding-space-comparison.ipynb`) and runs LogReg / XGBoost / MLP on each fine-tuned model × pooling combination.

**No encoder inference needed** — arrays are already on disk.

This tests whether the geometry the encoder learned during fine-tuning is sufficient for a shallow classifier to match end-to-end fine-tuned BERT performance (~87.7% binary, ~TBD 3-class).

In [8]:
# ── Load pre-computed embeddings from the comparison pkl ──────────────────────
import pickle

PKL_PATH = 'embeddings/embedding_comparison_arrays.pkl'
with open(PKL_PATH, 'rb') as f:
    all_arrays = pickle.load(f)

# Show what's available
print("Available model × pooling combos:")
for model_name, poolings in all_arrays.items():
    safe_name = model_name.replace('\n', ' ')
    for pool in poolings:
        shape = poolings[pool]['X_train'].shape
        labels = sorted(set(poolings[pool]['y_train'].tolist()))
        print(f"  {safe_name:40s} [{pool}]  X_train={shape}  labels={labels}")

Available model × pooling combos:
  roberta-base (pre-trained)               [cls]  X_train=(15383, 768)  labels=[0, 1, 2]
  roberta-base (pre-trained)               [rationale]  X_train=(15383, 768)  labels=[0, 1, 2]
  bert-base (pre-trained)                  [cls]  X_train=(15383, 768)  labels=[0, 1, 2]
  bert-base (pre-trained)                  [rationale]  X_train=(15383, 768)  labels=[0, 1, 2]
  bert-binary (fine-tuned)                 [cls]  X_train=(15383, 768)  labels=[0, 1]
  bert-binary (fine-tuned)                 [rationale]  X_train=(15383, 768)  labels=[0, 1]
  bert-3class (fine-tuned)                 [cls]  X_train=(15383, 768)  labels=[0, 1, 2]
  roberta-3class (fine-tuned)              [cls]  X_train=(15383, 768)  labels=[0, 1, 2]


In [9]:
# ── Route B: Run classifiers on ALL embedding sources ─────────────────────────
# Includes fine-tuned AND vanilla (pre-trained) models for a single combined table.
#
# The pkl key names contain newlines — we match on the prefix.

from copy import deepcopy
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from xgboost import XGBClassifier

LABEL_MAP_2 = {0: 'hate', 1: 'non-hate'}
LABEL_MAP_3 = {0: 'hate', 1: 'offensive', 2: 'normal'}

# (prefix_in_pkl, pooling, label_map, display_name)
ROUTE_B_TARGETS = [
    ('bert-binary',    'cls',       LABEL_MAP_2, 'Fine-Tuned BERT Binary [cls]'),
    ('bert-binary',    'rationale', LABEL_MAP_2, 'Fine-Tuned BERT Binary [rationale]'),
    ('bert-3class',    'cls',       LABEL_MAP_3, 'Fine-Tuned BERT 3-Class [cls]'),
    ('roberta-3class', 'cls',       LABEL_MAP_3, 'Fine-Tuned RoBERTa 3-Class [cls]'),
    ('bert-base',      'cls',       LABEL_MAP_3, 'Vanilla BERT [cls]'),
    ('roberta-base',   'cls',       LABEL_MAP_3, 'Vanilla RoBERTa [cls]'),
]

def get_arrays(prefix, pooling):
    """Look up arrays by model name prefix (handles the newline in keys)."""
    for key in all_arrays:
        if key.startswith(prefix):
            if pooling in all_arrays[key]:
                return all_arrays[key][pooling]
    raise KeyError(f"No match for prefix={prefix!r}, pooling={pooling!r}")


def make_classifiers(num_classes):
    """Build classifier dict, adapting eval_metric for binary vs multiclass."""
    metric = 'logloss' if num_classes == 2 else 'mlogloss'
    return {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42,
                                                   class_weight='balanced'),
        'XGBoost':             XGBClassifier(n_estimators=200, learning_rate=0.05,
                                              max_depth=6, eval_metric=metric,
                                              random_state=42),
        'MLP (256→128→64)':    MLPClassifier(hidden_layer_sizes=(256, 128, 64),
                                              activation='relu', max_iter=500,
                                              random_state=42, early_stopping=True,
                                              validation_fraction=0.1),
    }


# ── Run everything ───────────────────────────────────────────────────────────
all_route_b_rows = []

for prefix, pooling, lmap, display_name in ROUTE_B_TARGETS:
    data = get_arrays(prefix, pooling)
    X_tr, X_te = data['X_train'], data['X_test']
    y_tr, y_te = data['y_train'], data['y_test']
    num_classes = len(set(y_tr.tolist()))
    tnames = [lmap[k] for k in sorted(lmap)]

    print(f"\n{'='*60}")
    print(f"  {display_name}  ({num_classes}-class)")
    print(f"{'='*60}")
    print(f"  X_train={X_tr.shape}  X_test={X_te.shape}")
    print(f"  Train label dist: {dict(zip(*np.unique(y_tr, return_counts=True)))}")

    classifiers = make_classifiers(num_classes)
    for clf_name, clf in classifiers.items():
        m = deepcopy(clf)
        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_te)
        acc = accuracy_score(y_te, y_pred)
        mf1 = f1_score(y_te, y_pred, average='macro')
        print(f"\n  ── {clf_name} ──  acc={acc:.4f}  macro_f1={mf1:.4f}")
        print(classification_report(y_te, y_pred, target_names=tnames))
        all_route_b_rows.append({
            'Embeddings': display_name,
            'Task': f"{num_classes}-class",
            'Classifier': clf_name,
            'Accuracy': round(acc, 4),
            'Macro F1': round(mf1, 4),
        })


  Fine-Tuned BERT Binary [cls]  (2-class)
  X_train=(15383, 768)  X_test=(1924, 768)
  Train label dist: {np.int64(0): np.int64(4748), np.int64(1): np.int64(10635)}

  ── Logistic Regression ──  acc=0.8290  macro_f1=0.8109
              precision    recall  f1-score   support

        hate       0.68      0.84      0.75       594
    non-hate       0.92      0.82      0.87      1330

    accuracy                           0.83      1924
   macro avg       0.80      0.83      0.81      1924
weighted avg       0.85      0.83      0.83      1924


  ── XGBoost ──  acc=0.8446  macro_f1=0.8172
              precision    recall  f1-score   support

        hate       0.75      0.74      0.75       594
    non-hate       0.88      0.89      0.89      1330

    accuracy                           0.84      1924
   macro avg       0.82      0.82      0.82      1924
weighted avg       0.84      0.84      0.84      1924


  ── MLP (256→128→64) ──  acc=0.8404  macro_f1=0.8159
              precisi

In [ ]:
# ── Combined summary table ────────────────────────────────────────────────────
route_b_df = pd.DataFrame(all_route_b_rows)

# Add reference baseline: end-to-end fine-tuned BERT (Route A)
reference_rows = [
    {'Embeddings': 'Fine-Tuned BERT + Inference on BERT',
     'Task': '2-class', 'Classifier': '(end-to-end)', 'Accuracy': 0.877, 'Macro F1': 0.874},
]
ref_df = pd.DataFrame(reference_rows)
combined_df = pd.concat([route_b_df, ref_df], ignore_index=True)

# Sort: 2-class first, then 3-class; within each group by Macro F1 descending
combined_df['_sort'] = combined_df['Task'].map({'2-class': 0, '3-class': 1})
combined_df = combined_df.sort_values(['_sort', 'Macro F1'], ascending=[True, False]).drop(columns='_sort')

print("="*85)
print("  ALL RESULTS — Embeddings + Shallow Classifiers")
print("="*85)
print(combined_df[['Embeddings', 'Task', 'Classifier', 'Accuracy', 'Macro F1']].to_string(index=False))

# Highlight key comparisons
print("\n── Key takeaways ──")
best_binary = combined_df[combined_df['Task'] == '2-class'].iloc[0]
print(f"  Best Route B (binary) : {best_binary['Embeddings']} + {best_binary['Classifier']}")
print(f"                          acc={best_binary['Accuracy']:.4f}  macro_f1={best_binary['Macro F1']:.4f}")
print(f"  End-to-end BERT       : acc=0.877   macro_f1=0.874")
print(f"  Gap                   : {0.874 - best_binary['Macro F1']:.4f} macro F1")

best_3class = combined_df[(combined_df['Task'] == '3-class') & (combined_df['Classifier'] != '(end-to-end)')].iloc[0]
vanilla_best = combined_df[(combined_df['Task'] == '3-class') & (combined_df['Embeddings'].str.startswith('Vanilla'))].iloc[0]
print(f"\n  Best fine-tuned (3-cls) : {best_3class['Embeddings']} + {best_3class['Classifier']}")
print(f"                            acc={best_3class['Accuracy']:.4f}  macro_f1={best_3class['Macro F1']:.4f}")
print(f"  Best vanilla (3-cls)    : {vanilla_best['Embeddings']} + {vanilla_best['Classifier']}")
print(f"                            acc={vanilla_best['Accuracy']:.4f}  macro_f1={vanilla_best['Macro F1']:.4f}")
print(f"  Improvement             : +{best_3class['Macro F1'] - vanilla_best['Macro F1']:.4f} macro F1")

  ROUTE B SUMMARY — Shallow classifiers on fine-tuned embeddings

── 2-class ──
             Embeddings          Classifier  Accuracy  Macro F1
bert-binary [rationale]             XGBoost    0.8649    0.8403
bert-binary [rationale]    MLP (256→128→64)    0.8571    0.8346
bert-binary [rationale] Logistic Regression    0.8467    0.8286
      bert-binary [cls]             XGBoost    0.8446    0.8172
      bert-binary [cls]    MLP (256→128→64)    0.8404    0.8159
      bert-binary [cls] Logistic Regression    0.8290    0.8109

── 3-class ──
          Embeddings          Classifier  Accuracy  Macro F1
roberta-3class [cls]             XGBoost    0.7027    0.6907
   bert-3class [cls]             XGBoost    0.6949    0.6839
roberta-3class [cls] Logistic Regression    0.6881    0.6808
roberta-3class [cls]    MLP (256→128→64)    0.6913    0.6758
   bert-3class [cls]    MLP (256→128→64)    0.6757    0.6616
   bert-3class [cls] Logistic Regression    0.6601    0.6531

── Reference baselines ──
  P